In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [5]:
from scripts.helpers import save_classification_report

RANDOM_STATE = 42

In [6]:
df = pd.read_csv(PROJECT_ROOT / "working_data" / "nhamcs_data_2018_22.csv")
df_nlp = pd.read_csv(PROJECT_ROOT / "working_data" / "nlp_oof_logits_probs.csv")
df_emergency = pd.read_csv(PROJECT_ROOT / "working_data" / "nhamcs_emergency_keyword_flags_matched_only.csv")

In [7]:
df.columns

Index(['visit_month', 'day_of_week', 'arrival_time', 'ems_arrival',
       'vitals_during_visit', 'age', 'residence', 'sex', 'insurance',
       'no_payment', 'region', 'temp', 'heart_rate', 'resp_rate', 'sys_bp',
       'dias_bp', 'spo2', 'pain_score', 'seen_last_72h', 'episode',
       'is_injury_poison', 'hist_alzheimers', 'hist_asthma', 'hist_cancer',
       'hist_stroke', 'hist_ckd', 'hist_copd', 'hist_chf', 'hist_cad',
       'hist_depression', 'hist_diabetes_t1', 'hist_diabetes_t2',
       'hist_diabetes_unspec', 'hist_esrd', 'hist_pe', 'hist_hiv',
       'hist_high_cholesterol', 'hist_hypertension', 'hist_obesity',
       'hist_sleep_apnea', 'hist_osteoporosis', 'hist_substance_abuse',
       'intervention_iv_fluids', 'target_triage_acuity', 'wait_time_minutes',
       'race', 'year', 'chief_complaint_text', 'injury_cause_text'],
      dtype='str')

In [8]:
import numpy as np

print("Applying Cyclical Encoding to Time Features...")

# 1. Clean and convert arrival_time (HHMM integer) to a continuous hour scale (0 to 23.99)
# Example: 1430 -> 14 hours + (30/60) minutes = 14.5
if 'arrival_time' in df.columns:
    # Set missing or invalid times (often negative in NHAMCS) to NaN    
    df['arrival_time'] = np.where(df['arrival_time'] < 0, np.nan, df['arrival_time'])
    
    # Extract hours and minutes
    hours = np.floor_divide(df['arrival_time'], 100)
    minutes = np.mod(df['arrival_time'], 100)
    
    # Create the continuous hour feature
    df['arrival_hour_float'] = hours + (minutes / 60.0)
    df['arrival_hour'] = hours
    
    # Shift overlap flag (06:00-08:00, 18:00-20:00)
    df['is_shift_change'] = (
        ((df['arrival_hour_float'] >= 6) & (df['arrival_hour_float'] < 8))
        | ((df['arrival_hour_float'] >= 18) & (df['arrival_hour_float'] < 20))
    ).astype(int)

# 2. Define the exact maximum values for a full cycle
cycle_maxes = {
    'visit_month': 12.0,
    'day_of_week': 7.0,
    'arrival_hour_float': 24.0
}

# 3. Apply the Sine and Cosine Transformations
for col, max_val in cycle_maxes.items():
    if col in df.columns:
        # Calculate the angle on the circle
        angle = 2 * np.pi * df[col] / max_val
        
        # Create the Sin and Cos features
        df[f'{col}_sin'] = np.sin(angle)
        df[f'{col}_cos'] = np.cos(angle)

# Weekend/weeknight interaction
if 'day_of_week' in df.columns:
    max_dow = df['day_of_week'].max()
    weekend_days = [5, 6] if max_dow <= 6 else [6, 7]
    is_weekend = df['day_of_week'].isin(weekend_days)
    if 'arrival_hour_float' in df.columns:
        df['is_weekend_night'] = (is_weekend & (df['arrival_hour_float'] >= 18)).astype(int)
    else:
        df['is_weekend'] = is_weekend.astype(int)

# 4. Drop the original linear time columns to prevent multicollinearity
cols_to_drop = ['visit_month', 'day_of_week', 'arrival_time', 'arrival_hour_float']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print(f"Dataframe shape after cyclical encoding: {df.shape}")
print("\nSample of the new Time Features:")
display(df[[c for c in df.columns if '_sin' in c or '_cos' in c]].head())

Applying Cyclical Encoding to Time Features...
Dataframe shape after cyclical encoding: (58124, 55)

Sample of the new Time Features:


,visit_month_sin,visit_month_cos,day_of_week_sin,day_of_week_cos,arrival_hour_float_sin,arrival_hour_float_cos
0,-2.449294e-16,1.000000,0.781831,0.623490,-0.748956,0.662620
1,-2.449294e-16,1.000000,0.781831,0.623490,-0.957571,0.288196
2,-2.449294e-16,1.000000,-0.781831,0.623490,-0.625923,-0.779884
3,-2.449294e-16,1.000000,-0.433884,-0.900969,-0.994522,-0.104528
4,-5.000000e-01,0.866025,0.974928,-0.222521,-0.983255,-0.182236


In [9]:
print("Adding clinical ratios and vital missingness...")

df["shock_index"] = df["heart_rate"] / df["sys_bp"].replace(0, np.nan)
df["shock_index_age_adj"] = df["shock_index"] * np.where(df["age"] >= 65, 1.2, 1.0)

df["map"] = (df["sys_bp"] + 2 * df["dias_bp"]) / 3
df["pulse_pressure"] = df["sys_bp"] - df["dias_bp"]

df["age_hr_interaction"] = df["age"] * df["heart_rate"]

df["resp_spo2_ratio"] = df["resp_rate"] / df["spo2"].replace(0, np.nan)

df["elderly_tachy"] = ((df["age"] >= 65) & (df["heart_rate"] > 100)).astype(int)

history_cols = [c for c in df.columns if any(k in c for k in "hist_")]
if history_cols:
    hist_numeric = df[history_cols].apply(pd.to_numeric, errors="coerce")
    df["history_count"] = hist_numeric.fillna(0).sum(axis=1)

# NEWS2-style score (approximate)
news2 = pd.Series(0, index=df.index, dtype=float)
if "resp_rate" in df.columns:
    rr = df["resp_rate"]
    news2 += np.select([
        rr <= 8,
        (rr >= 9) & (rr <= 11),
        (rr >= 12) & (rr <= 20),
        (rr >= 21) & (rr <= 24),
        rr >= 25,
    ], [3, 1, 0, 2, 3], default=0)
if "spo2" in df.columns:
    sp = df["spo2"]
    news2 += np.select([
        sp <= 91,
        (sp >= 92) & (sp <= 93),
        (sp >= 94) & (sp <= 95),
        sp >= 96,
    ], [3, 2, 1, 0], default=0)
if "temp" in df.columns:
    t = df["temp"]
    news2 += np.select([
        t <= 35.0,
        (t > 35.0) & (t <= 36.0),
        (t > 36.0) & (t <= 38.0),
        (t > 38.0) & (t <= 39.0),
        t > 39.0,
    ], [3, 1, 0, 1, 2], default=0)
if "sys_bp" in df.columns:
    sbp = df["sys_bp"]
    news2 += np.select([
        sbp <= 90,
        (sbp >= 91) & (sbp <= 100),
        (sbp >= 101) & (sbp <= 110),
        (sbp >= 111) & (sbp <= 219),
        sbp >= 220,
    ], [3, 2, 1, 0, 3], default=0)
if "heart_rate" in df.columns:
    hr = df["heart_rate"]
    news2 += np.select([
        hr <= 40,
        (hr >= 41) & (hr <= 50),
        (hr >= 51) & (hr <= 90),
        (hr >= 91) & (hr <= 110),
        (hr >= 111) & (hr <= 130),
        hr >= 131,
    ], [3, 1, 0, 1, 2, 3], default=0)


df["news2_score"] = news2

vital_cols = ["heart_rate", "sys_bp", "dias_bp", "resp_rate", "spo2", "temp"]
vital_cols = [c for c in vital_cols if c in df.columns]
if vital_cols:
    df["vital_missing"] = df[vital_cols].isna().any(axis=1).astype(int)
    for col in vital_cols:
        df[f"{col}_missing"] = df[col].isna().astype(int)

num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

Adding clinical ratios and vital missingness...


In [10]:
if "row_index" in df_nlp.columns:
    df_nlp = (
        df_nlp
        .sort_values("row_index")
        .set_index("row_index")
        .reindex(df.index)
        .reset_index(drop=True)
    )

df_nlp = df_nlp.drop(columns=["row_index", "fold", "year"], errors="ignore")
df_emergency = df_emergency.drop(columns=["row_index"], errors="ignore")

if len(df_nlp) != len(df):
    raise ValueError("NLP features are not aligned with main dataframe length.")

In [11]:
# Prepare features and target for ordinal triage
cols_to_drop = [
    "intervention_iv_fluids",
    "vitals_during_visit",
    "wait_time_minutes",
    "year",
    "target_triage_acuity",
    "chief_complaint_text",
    "injury_cause_text",
    "true_class",
    "pred_class",
]
X_tabular = df.drop(columns=[c for c in cols_to_drop if c in df.columns]).copy()
year_groups = df["year"].copy()

def map_esi_to_group(val: int):
    if val in (1, 2):
        return 0
    if val == 3:
        return 1
    if val in (4, 5):
        return 2
    return np.nan

y = df["target_triage_acuity"].map(map_esi_to_group)
y = y.dropna().astype(int)
X_tabular = X_tabular.loc[y.index].copy()
year_groups = year_groups.loc[y.index].copy()

def to_category(df_in):
    df_out = df_in.copy()
    obj_cols = df_out.select_dtypes(include=["object", "string"]).columns
    for col in obj_cols:
        df_out[col] = df_out[col].astype("category")
    return df_out

X_tabular = to_category(X_tabular)

hist_cols = [c for c in X_tabular.columns if c.startswith("hist")]
if hist_cols:
    X_tabular[hist_cols] = X_tabular[hist_cols].fillna(0).astype(int).astype(bool)

X_nlp = df_nlp.loc[y.index].copy()
X_emergency = to_category(df_emergency.loc[y.index].copy())

In [13]:
# 1) Exact sensitive/proxy names we want removed if present
bias_columns = {
    "residence", "region", "race", "no_payment", "insurance",
}


X_tabular = X_tabular.drop(columns=bias_columns, errors="ignore")

print("Bias exclusion applied.")
print(f"Columns removed: {len(bias_columns)}")
if bias_columns:
    print(bias_columns)
print(f"X shape after bias exclusion: {X_tabular.shape}")


Bias exclusion applied.
Columns removed: 5
{'residence', 'insurance', 'race', 'no_payment', 'region'}
X shape after bias exclusion: (58124, 59)


In [14]:
from sklearn.metrics import cohen_kappa_score

def qwk_score(y_true, y_pred):
    """Quadratic Weighted Kappa for ordinal triage evaluation."""
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")


In [15]:
import warnings
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
import xgboost as xgb
import lightgbm as lgb

from imblearn.ensemble import BalancedRandomForestClassifier

def compute_sample_weights(y_train):
    classes = np.unique(y_train)
    weights = compute_class_weight("balanced", classes=classes, y=y_train)
    weight_map = dict(zip(classes, weights))
    return np.array([weight_map[val] for val in y_train])

def fit_cutpoints(y_true, y_pred, n_classes):
    df_cut = pd.DataFrame({"y": y_true, "pred": y_pred})
    medians = df_cut.groupby("y")["pred"].median().sort_index()
    if len(medians) < n_classes:
        return np.quantile(y_pred, np.linspace(0, 1, n_classes + 1)[1:-1])
    cutpoints = []
    for i in range(n_classes - 1):
        cutpoints.append((medians.iloc[i] + medians.iloc[i + 1]) / 2.0)
    return np.array(cutpoints)

def apply_cutpoints(y_pred, cutpoints):
    return np.digitize(y_pred, cutpoints, right=False)

def category_codes(df_in):
    df_out = df_in.copy()
    cat_cols = df_out.select_dtypes(include=["category"]).columns
    for col in cat_cols:
        df_out[col] = df_out[col].cat.codes
    return df_out.fillna(-1)

y_ord = y.astype(int).copy()
n_classes = int(y_ord.nunique())

X_tabular_codes = category_codes(X_tabular)
X_emergency_codes = category_codes(X_emergency)

years_sorted = np.sort(year_groups.unique())
year_bins = np.array_split(years_sorted, 3)
year_bucket_map = {year: idx for idx, years in enumerate(year_bins) for year in years}
year_bucket = year_groups.map(year_bucket_map)
splitter = GroupKFold(n_splits=3)

def eval_regressor_with_cutpoints(model, X, y, name):
    oof_class = np.zeros(len(y), dtype=int)
    groups = year_bucket.loc[X.index]
    for fold, (train_idx, val_idx) in enumerate(splitter.split(X, y, groups=groups), start=1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        sample_weight = compute_sample_weights(y_train)
        model.fit(X_train, y_train, sample_weight=sample_weight)
        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)
        cutpoints = fit_cutpoints(y_train, train_pred, n_classes)
        val_class = apply_cutpoints(val_pred, cutpoints)
        oof_class[val_idx] = val_class
        fold_qwk = qwk_score(y_val, val_class)
        print(f"{name} fold {fold} qwk: {fold_qwk:.4f}")
    oof_qwk = qwk_score(y, oof_class)
    print(f"{name} OOF qwk: {oof_qwk:.4f}")
    return oof_class

def eval_balanced_rf(model, X, y, name):
    oof_class = np.zeros(len(y), dtype=int)
    class_values = np.arange(n_classes)
    groups = year_bucket.loc[X.index]
    for fold, (train_idx, val_idx) in enumerate(splitter.split(X, y, groups=groups), start=1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        sample_weight = compute_sample_weights(y_train)
        model.fit(X_train, y_train, sample_weight=sample_weight)
        train_proba = model.predict_proba(X_train)
        val_proba = model.predict_proba(X_val)
        train_pred = train_proba @ class_values
        val_pred = val_proba @ class_values
        cutpoints = fit_cutpoints(y_train, train_pred, n_classes)
        val_class = apply_cutpoints(val_pred, cutpoints)
        oof_class[val_idx] = val_class
        fold_qwk = qwk_score(y_val, val_class)
        print(f"{name} fold {fold} qwk: {fold_qwk:.4f}")
    oof_qwk = qwk_score(y, oof_class)
    print(f"{name} OOF qwk: {oof_qwk:.4f}")
    return oof_class

xgb_reg = xgb.XGBRegressor(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
    enable_categorical=True,
)

lgb_reg = lgb.LGBMRegressor(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
)

brf = BalancedRandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

RUN_BASE_CV = False
if RUN_BASE_CV:
    print("Running year-wise CV with QWK and ordinal cutpoints...")
    _ = eval_regressor_with_cutpoints(xgb_reg, X_tabular, y_ord, "XGBRegressor")
    _ = eval_regressor_with_cutpoints(lgb_reg, X_tabular, y_ord, "LGBMRegressor")
    _ = eval_balanced_rf(brf, X_tabular_codes, y_ord, "BalancedRandomForest")

In [16]:
def eval_proba_classifier(model, X, y, name):
    oof_class = np.zeros(len(y), dtype=int)
    class_values = np.arange(n_classes)
    groups = year_groups.loc[X.index]
    for fold, (train_idx, val_idx) in enumerate(splitter.split(X, y, groups=groups), start=1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        sample_weight = compute_sample_weights(y_train)
        model.fit(X_train, y_train, sample_weight=sample_weight)
        train_proba = model.predict_proba(X_train)
        val_proba = model.predict_proba(X_val)
        train_pred = train_proba @ class_values
        val_pred = val_proba @ class_values
        cutpoints = fit_cutpoints(y_train, train_pred, n_classes)
        val_class = apply_cutpoints(val_pred, cutpoints)
        oof_class[val_idx] = val_class
        fold_qwk = qwk_score(y_val, val_class)
        print(f"{name} fold {fold} qwk: {fold_qwk:.4f}")
    oof_qwk = qwk_score(y, oof_class)
    print(f"{name} OOF qwk: {oof_qwk:.4f}")
    return oof_class

In [17]:
import mord

ordinal_at = mord.LogisticAT(alpha=1.0)
ordinal_it = mord.LogisticIT(alpha=1.0)

if RUN_BASE_CV:
    print("Running year-wise CV on X_emergency with ordinal models (mord)...")
    _ = eval_proba_classifier(ordinal_at, X_emergency_codes, y_ord, "Ordinal Logistic (AT)")
    _ = eval_proba_classifier(ordinal_it, X_emergency_codes, y_ord, "Ordinal Logistic (IT)")

In [18]:
from sklearn.linear_model import SGDClassifier

sgd_log = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-4,
    class_weight="balanced",
    max_iter=1000,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    )

if RUN_BASE_CV:
    print("Running year-wise CV on X_emergency with sparse-friendly SGD...")
    _ = eval_proba_classifier(sgd_log, X_emergency_codes, y_ord, "SGD (log_loss)")

In [19]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

linear_svc = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    )

linear_svc_cal = CalibratedClassifierCV(
    estimator=linear_svc,
    method="sigmoid",
    cv=3,
    )

if RUN_BASE_CV:
    print("Running year-wise CV on X_emergency with LinearSVC (calibrated)...")
    _ = eval_proba_classifier(linear_svc_cal, X_emergency_codes, y_ord, "LinearSVC (calibrated)")

In [20]:
# Extract OOF class probabilities with year-wise folds
from sklearn.base import clone

X_nlp_num = X_nlp.apply(pd.to_numeric, errors="coerce").fillna(0)

def regressor_soft_proba_from_preds(train_pred, val_pred, y_train, n_classes):
    centers = pd.Series(train_pred).groupby(y_train).median()
    if len(centers) < n_classes:
        centers = pd.Series(
            np.linspace(np.min(train_pred), np.max(train_pred), n_classes),
            index=range(n_classes),
        )
    centers = centers.reindex(range(n_classes)).interpolate().to_numpy()
    tau = np.var(train_pred) + 1e-6
    dist = (val_pred[:, None] - centers[None, :]) ** 2
    probs = np.exp(-dist / (2.0 * tau))
    return probs / probs.sum(axis=1, keepdims=True)

base_oof = {
    "xgb": np.zeros((len(y_ord), n_classes)),
    "lgb": np.zeros((len(y_ord), n_classes)),
    "brf": np.zeros((len(y_ord), n_classes)),
    "ord_at": np.zeros((len(y_ord), n_classes)),
    "ord_it": np.zeros((len(y_ord), n_classes)),
    "sgd": np.zeros((len(y_ord), n_classes)),
    "linear_svc": np.zeros((len(y_ord), n_classes)),
}
for fold, (train_idx, val_idx) in enumerate(splitter.split(X_tabular, y_ord, groups=year_bucket), start=1):
    X_tab_train, X_tab_val = X_tabular.iloc[train_idx], X_tabular.iloc[val_idx]
    X_tab_codes_train, X_tab_codes_val = X_tabular_codes.iloc[train_idx], X_tabular_codes.iloc[val_idx]
    X_em_train, X_em_val = X_emergency_codes.iloc[train_idx], X_emergency_codes.iloc[val_idx]
    y_train = y_ord.iloc[train_idx]
    sample_weight = compute_sample_weights(y_train)

    xgb_fold = clone(xgb_reg)
    xgb_fold.fit(X_tab_train, y_train, sample_weight=sample_weight)
    xgb_train_pred = xgb_fold.predict(X_tab_train)
    xgb_val_pred = xgb_fold.predict(X_tab_val)
    base_oof["xgb"][val_idx] = regressor_soft_proba_from_preds(
        xgb_train_pred, xgb_val_pred, y_train, n_classes
    )

    lgb_fold = clone(lgb_reg)
    lgb_fold.fit(X_tab_train, y_train, sample_weight=sample_weight)
    lgb_train_pred = lgb_fold.predict(X_tab_train)
    lgb_val_pred = lgb_fold.predict(X_tab_val)
    base_oof["lgb"][val_idx] = regressor_soft_proba_from_preds(
        lgb_train_pred, lgb_val_pred, y_train, n_classes
    )

    brf_fold = clone(brf)
    brf_fold.fit(X_tab_codes_train, y_train, sample_weight=sample_weight)
    base_oof["brf"][val_idx] = brf_fold.predict_proba(X_tab_codes_val)

    ordinal_at_fold = clone(ordinal_at)
    ordinal_it_fold = clone(ordinal_it)
    ordinal_at_fold.fit(X_em_train, y_train)
    ordinal_it_fold.fit(X_em_train, y_train)
    base_oof["ord_at"][val_idx] = ordinal_at_fold.predict_proba(X_em_val)
    base_oof["ord_it"][val_idx] = ordinal_it_fold.predict_proba(X_em_val)

    sgd_fold = clone(sgd_log)
    sgd_fold.fit(X_em_train, y_train, sample_weight=sample_weight)
    base_oof["sgd"][val_idx] = sgd_fold.predict_proba(X_em_val)

    linear_svc_fold = clone(linear_svc_cal)
    linear_svc_fold.fit(X_em_train, y_train, sample_weight=sample_weight)
    base_oof["linear_svc"][val_idx] = linear_svc_fold.predict_proba(X_em_val)

print("Base OOF probabilities ready:", sorted(base_oof.keys()))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002548 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2894
[LightGBM] [Info] Number of data points in the train set: 30326, number of used features: 58
[LightGBM] [Info] Start training from score 1.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003812 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2928
[LightGBM] [Info] Number of data points in the train set: 38005, number of used features: 58
[LightGBM] [Info] Start training from score 1.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006103 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enoug

In [21]:
# Meta-learner using base probabilities + NLP features
meta_X = np.hstack([
    base_oof["xgb"],
    base_oof["lgb"],
    base_oof["brf"],
    base_oof["ord_at"],
    base_oof["ord_it"],
    base_oof["sgd"],
    base_oof["linear_svc"],
    X_nlp_num.to_numpy(),
])

meta_oof = np.zeros(len(y_ord), dtype=int)
class_values = np.arange(n_classes)

for fold, (train_idx, val_idx) in enumerate(splitter.split(meta_X, y_ord, groups=year_bucket), start=1):
    X_train, X_val = meta_X[train_idx], meta_X[val_idx]
    y_train, y_val = y_ord.iloc[train_idx], y_ord.iloc[val_idx]
    sample_weight = compute_sample_weights(y_train)

    meta_model = LogisticRegression(
        solver="saga",
        max_iter=500,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    meta_model.fit(X_train, y_train, sample_weight=sample_weight)
    train_proba = meta_model.predict_proba(X_train)
    val_proba = meta_model.predict_proba(X_val)
    train_pred = train_proba @ class_values
    val_pred = val_proba @ class_values
    cutpoints = fit_cutpoints(y_train, train_pred, n_classes)
    val_class = apply_cutpoints(val_pred, cutpoints)
    meta_oof[val_idx] = val_class
    fold_qwk = qwk_score(y_val, val_class)
    print(f"Meta fold {fold} qwk: {fold_qwk:.4f}")

meta_qwk = qwk_score(y_ord, meta_oof)
print(f"Meta OOF qwk: {meta_qwk:.4f}")

Meta fold 1 qwk: 0.4871
Meta fold 2 qwk: 0.5023
Meta fold 3 qwk: 0.5130
Meta OOF qwk: 0.4965


In [ ]:
from sklearn.linear_model import LogisticRegression

def eval_meta_variant(params, X, y, name):
    oof_class = np.zeros(len(y), dtype=int)
    class_values = np.arange(n_classes)
    for fold, (train_idx, val_idx) in enumerate(splitter.split(X, y, groups=year_bucket), start=1):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        sample_weight = compute_sample_weights(y_train)

        model = LogisticRegression(**params)
        model.fit(X_train, y_train, sample_weight=sample_weight)
        train_proba = model.predict_proba(X_train)
        val_proba = model.predict_proba(X_val)
        train_pred = train_proba @ class_values
        val_pred = val_proba @ class_values
        cutpoints = fit_cutpoints(y_train, train_pred, n_classes)
        val_class = apply_cutpoints(val_pred, cutpoints)
        oof_class[val_idx] = val_class
        fold_qwk = qwk_score(y_val, val_class)
        print(f"{name} fold {fold} qwk: {fold_qwk:.4f}")
    oof_qwk = qwk_score(y, oof_class)
    print(f"{name} OOF qwk: {oof_qwk:.4f}")
    return oof_qwk

variant_grid = [
    {"name": "l2_C0.25", "params": {
        "solver": "saga", "max_iter": 800, "class_weight": "balanced", "n_jobs": -1, "random_state": RANDOM_STATE,
        "c "penalty": "l2",
    }},
    {"name": "l2_C1.0", "params": {
        "solver": "saga", "max_iter": 800, "class_weight": "balanced", "n_jobs": -1, "random_state": RANDOM_STATE,
        "C": 1.0, "penalty": "l2",
    }},
    {"name": "l2_C4.0", "params": {
        "solver": "saga", "max_iter": 800, "class_weight": "balanced", "n_jobs": -1, "random_state": RANDOM_STATE,
        "C": 4.0, "penalty": "l2",
    }},
    {"name": "elastic_C1.0_l1r0.1", "params": {
        "solver": "saga", "max_iter": 1200, "class_weight": "balanced", "n_jobs": -1, "random_state": RANDOM_STATE,
        "C": 1.0, "penalty": "elasticnet", "l1_ratio": 0.1,
    }},
    {"name": "elastic_C1.0_l1r0.5", "params": {
        "solver": "saga", "max_iter": 1200, "class_weight": "balanced", "n_jobs": -1, "random_state": RANDOM_STATE,
        "C": 1.0, "penalty": "elasticnet", "l1_ratio": 0.5,
    }},
    {"name": "elastic_C1.0_l1r0.9", "params": {
        "solver": "saga", "max_iter": 1200, "class_weight": "balanced", "n_jobs": -1, "random_state": RANDOM_STATE,
        "C": 1.0, "penalty": "elasticnet", "l1_ratio": 0.9,
    }},
]

variant_scores = {}
for item in variant_grid:
    name = item["name"]
    print("\n===", name, "===")
    score = eval_meta_variant(item["params"], meta_X, y_ord, name)
    variant_scores[name] = score

print("\nMeta-learner variant summary (sorted by OOF QWK):")
for name, score in sorted(variant_scores.items(), key=lambda x: x[1], reverse=True):
    print(f"{name}: {score:.4f}")


=== l2_C0.25 ===
l2_C0.25 fold 1 qwk: 0.5004
l2_C0.25 fold 2 qwk: 0.5025
l2_C0.25 fold 3 qwk: 0.5128
l2_C0.25 OOF qwk: 0.5037

=== l2_C1.0 ===
l2_C1.0 fold 1 qwk: 0.4871
l2_C1.0 fold 2 qwk: 0.5023
l2_C1.0 fold 3 qwk: 0.5130
l2_C1.0 OOF qwk: 0.4965

=== l2_C4.0 ===
l2_C4.0 fold 1 qwk: 0.4585
l2_C4.0 fold 2 qwk: 0.5012
l2_C4.0 fold 3 qwk: 0.5131
l2_C4.0 OOF qwk: 0.4808

=== elastic_C1.0_l1r0.1 ===
elastic_C1.0_l1r0.1 fold 1 qwk: 0.4876
elastic_C1.0_l1r0.1 fold 2 qwk: 0.5024
elastic_C1.0_l1r0.1 fold 3 qwk: 0.5131
elastic_C1.0_l1r0.1 OOF qwk: 0.4967

=== elastic_C1.0_l1r0.5 ===
elastic_C1.0_l1r0.5 fold 1 qwk: 0.4911
elastic_C1.0_l1r0.5 fold 2 qwk: 0.5028
elastic_C1.0_l1r0.5 fold 3 qwk: 0.5135
elastic_C1.0_l1r0.5 OOF qwk: 0.4988

=== elastic_C1.0_l1r0.9 ===
elastic_C1.0_l1r0.9 fold 1 qwk: 0.4951
elastic_C1.0_l1r0.9 fold 2 qwk: 0.5027
elastic_C1.0_l1r0.9 fold 3 qwk: 0.5127
elastic_C1.0_l1r0.9 OOF qwk: 0.5008

Meta-learner variant summary (sorted by OOF QWK):
l2_C0.25: 0.5037
elastic_C1.0_l1

In [24]:
from sklearn.base import clone
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils.validation import has_fit_parameter

# Tree-based meta learner and stacked meta learner (light variants)

def eval_meta_proba_model(model, X, y, name):
    oof_class = np.zeros(len(y), dtype=int)
    class_values = np.arange(n_classes)
    for fold, (train_idx, val_idx) in enumerate(splitter.split(X, y, groups=year_bucket), start=1):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        sample_weight = compute_sample_weights(y_train)

        model_fold = clone(model)
        if has_fit_parameter(model_fold, "sample_weight"):
            model_fold.fit(X_train, y_train, sample_weight=sample_weight)
        else:
            model_fold.fit(X_train, y_train)

        train_proba = model_fold.predict_proba(X_train)
        val_proba = model_fold.predict_proba(X_val)
        train_pred = train_proba @ class_values
        val_pred = val_proba @ class_values
        cutpoints = fit_cutpoints(y_train, train_pred, n_classes)
        val_class = apply_cutpoints(val_pred, cutpoints)
        oof_class[val_idx] = val_class
        fold_qwk = qwk_score(y_val, val_class)
        print(f"{name} fold {fold} qwk: {fold_qwk:.4f}")

    oof_qwk = qwk_score(y, oof_class)
    print(f"{name} OOF qwk: {oof_qwk:.4f}")
    return oof_qwk

lgb_meta = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
)

stack_base = [
    (
        "lr",
        LogisticRegression(
            solver="saga",
            max_iter=800,
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    ),
    ("lgb", lgb_meta),
]

stack_meta = StackingClassifier(
    estimators=stack_base,
    final_estimator=LogisticRegression(
        solver="lbfgs",
        max_iter=600,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    stack_method="predict_proba",
    passthrough=False,
    n_jobs=-1,
)

print("\nTree-based meta learner:")
_ = eval_meta_proba_model(lgb_meta, meta_X, y_ord, "Meta LGBM")

print("\nStacked meta learner:")
_ = eval_meta_proba_model(stack_meta, meta_X, y_ord, "Meta Stacking")


Tree-based meta learner:
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002123 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4160
[LightGBM] [Info] Number of data points in the train set: 30326, number of used features: 26
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
Meta LGBM fold 1 qwk: 0.4961
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001933 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4210
[LightGBM] [Info] Number of data points in the train set: 38005, number of used features: 26
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
Meta LGBM fold 2 qwk: 0.5021
[LightGBM] [In

In [ ]:
import json
from datetime import datetime, timezone
import joblib

# Fit final meta-learner on all data and save artifacts
final_meta_model = LogisticRegression(
    solver="saga",
    max_iter=500,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
 )
final_meta_model.fit(meta_X, y_ord, sample_weight=compute_sample_weights(y_ord))

artifact_dir = PROJECT_ROOT / "results" / "model_artifacts" / "meta_learner_v1"
artifact_dir.mkdir(parents=True, exist_ok=True)
model_path = artifact_dir / "meta_model.joblib"
meta_path = artifact_dir / "meta_model_metadata.json"

joblib.dump(final_meta_model, model_path)

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "model_type": "logistic_regression_meta",
    "n_classes": int(n_classes),
    "feature_blocks": ["xgb", "lgb", "brf", "ord_at", "ord_it", "sgd", "linear_svc", "nlp"],
    "year_bins": [list(map(int, b)) for b in year_bins],
    "class_mapping": {"0": "urgent", "1": "emergent", "2": "non_urgent"},
}

with meta_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved meta model: {model_path}")
print(f"Saved metadata: {meta_path}")

In [ ]:
from sklearn.metrics import classification_report, fbeta_score

print("Meta learner classification report:")
print(classification_report(y_ord, meta_oof, digits=4))

meta_f2 = fbeta_score(y_ord, meta_oof, beta=2, average="weighted")
print(f"Meta weighted F2: {meta_f2:.4f}")

Meta learner classification report:
              precision    recall  f1-score   support

           0     0.3630    0.6956    0.4771      9443
           1     0.6609    0.4494    0.5350     29568
           2     0.6344    0.6613    0.6476     19113

    accuracy                         0.5591     58124
   macro avg     0.5528    0.6021    0.5532     58124
weighted avg     0.6038    0.5591    0.5626     58124

Meta weighted F2: 0.5554
